# 07. Результати для презентації

Збирає CSV з ноутбуків 02-06 і робить графіки в `results/` для слайду Results.
Запускати після інших ноутбуків.

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.evaluate import summarize

def load(name):
    path = RESULTS / name
    if path.exists():
        return pd.read_csv(path)
    print("Немає", name, "- запусти відповідний ноутбук")
    return None

adaptive = load("04_adaptive.csv")
enhancers = load("03_enhancers.csv")
artelab = load("06_artelab.csv")
tuning = load("05_optuna.csv")

## Рисунок 1. Криві стійкості: raw / classic / adaptive / cascade (zxing)

In [ ]:
if adaptive is not None:
    data = adaptive[adaptive["decoder"] == "zxing"]
    curves = summarize(data, by=["method", "degradation", "level"])
    clean = curves[curves["degradation"] == "clean"]
    names = []
    for name in curves["degradation"].unique():
        if name != "clean":
            names.append(name)
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for ax, name in zip(axes.flat, names):
        for method_name in ["raw", "classic", "adaptive", "cascade"]:
            line = curves[(curves["degradation"] == name) & (curves["method"] == method_name)]
            start = clean[clean["method"] == method_name]
            ax.plot([0] + list(line["level"]),
                    list(start["decode_rate"]) + list(line["decode_rate"]),
                    marker="o", label=method_name)
        ax.set_title(name)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=0.3)
    axes[0, 0].legend()
    plt.tight_layout()
    plt.savefig(RESULTS / "fig1_robustness_curves.png", dpi=150)
    plt.show()

## Рисунок 2. Загальний decode rate і час на кадр

In [ ]:
if adaptive is not None:
    damaged = adaptive[adaptive["level"] > 0]
    overall = summarize(damaged, by=["method", "decoder"])
    grid = overall.pivot(index="method", columns="decoder", values="decode_rate")
    grid = grid.loc[["raw", "classic", "adaptive", "cascade"]]
    grid.plot(kind="bar", figsize=(8, 4), rot=0)
    plt.ylabel("decode rate (усі пошкодження, рівні 1-5)")
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS / "fig2_overall.png", dpi=150)
    plt.show()
    print(summarize(damaged, by=["method"])[["method", "decode_rate", "wrong_rate", "ms"]].round(3))

## Таблиця 1. Найкращий окремий метод для кожного пошкодження (zxing)

In [ ]:
if enhancers is not None:
    damaged = enhancers[(enhancers["level"] > 0) & (enhancers["decoder"] == "zxing")]
    rates = summarize(damaged, by=["degradation", "method"])
    best_rows = []
    for name in rates["degradation"].unique():
        part = rates[rates["degradation"] == name]
        raw_rate = part[part["method"] == "raw"]["decode_rate"].iloc[0]
        best = part.sort_values("decode_rate", ascending=False).iloc[0]
        best_rows.append({"degradation": name, "raw": round(raw_rate, 2),
                          "best_method": best["method"], "best": round(best["decode_rate"], 2)})
    print(pd.DataFrame(best_rows).to_string(index=False))

## Синтетика проти реальних фото (Arte-Lab)

In [ ]:
if artelab is not None and adaptive is not None:
    real = summarize(artelab[artelab["level"] > 0], by=["method"])[["method", "decode_rate"]]
    real.columns = ["method", "real (Arte-Lab)"]
    synthetic_part = adaptive[(adaptive["level"] > 0)
                              & adaptive["degradation"].isin(artelab["degradation"].unique())]
    synthetic = summarize(synthetic_part, by=["method"])[["method", "decode_rate"]]
    synthetic.columns = ["method", "synthetic"]
    print(synthetic.merge(real, on="method").round(3).to_string(index=False))

## Optuna: default проти tuned на test

In [ ]:
if tuning is not None:
    print(tuning.pivot_table(index="degradation", columns="params", values="correct", aggfunc="mean").round(3))